# ET Fundación — Data availability diagnostic tests

This notebook contains diagnostic tests for the ET downscaling data pipeline.

## Objectives

1. Verify safe handling of missing Sentinel-2 observations.
2. Verify that missing satellite data remain masked rather than becoming artificial values.
3. Inspect Sentinel-2 and Sentinel-1 data availability.
4. Evaluate spatial coverage independently from extraction.
5. Support later selection of scientifically defensible coverage thresholds.

Coverage thresholds used for modeling must not be imposed prematurely during data extraction.

In [1]:
from pathlib import Path
import sys


def find_repository_root(start_path):
    current_path = Path(start_path).resolve()

    for candidate in [
        current_path,
        *current_path.parents,
    ]:
        if (candidate / "src" / "et_downscaling").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the repository root."
    )


REPO_ROOT = find_repository_root(Path.cwd())
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )


print("Repository root:", REPO_ROOT)
print("Source path:", SRC_PATH)

Repository root: E:\1SIG22\Varios\github\git_github\26-et-downscaling-fundacion
Source path: E:\1SIG22\Varios\github\git_github\26-et-downscaling-fundacion\src


In [2]:
import ee


EE_PROJECT = "ee-change"

ee.Initialize(
    project=EE_PROJECT
)

print(
    "Earth Engine initialized with project:",
    EE_PROJECT,
)

c:\Users\usrlabsis22\AppData\Local\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Earth Engine initialized with project: ee-change


In [4]:
from et_downscaling.sentinel2 import (
    build_s2_medoid,
)


test_geometry = (
    ee.Geometry
    .Point(
        [
            -74.2,
            10.5,
        ]
    )
    .buffer(500)
)

empty_s2_collection = (
    ee.ImageCollection([])
)


empty_s2_medoid = (
    build_s2_medoid(
        empty_s2_collection,
        test_geometry,
    )
)


band_names = (
    empty_s2_medoid
    .bandNames()
    .getInfo()
)


statistics = (
    empty_s2_medoid
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=test_geometry,
        scale=20,
        maxPixels=1e6,
    )
    .getInfo()
)


print(
    "Bands:",
    band_names,
)

print(
    "Statistics:",
    statistics,
)

c:\Users\usrlabsis22\AppData\Local\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Bands: ['Blue', 'Green', 'Red', 'RedEdge1', 'RedEdge2', 'RedEdge3', 'NIR', 'SWIR1', 'SWIR2']
Statistics: {'Blue': None, 'Green': None, 'NIR': None, 'Red': None, 'RedEdge1': None, 'RedEdge2': None, 'RedEdge3': None, 'SWIR1': None, 'SWIR2': None}


In [6]:
from pathlib import Path
import sys
import ee

repo_root = Path.cwd().resolve()

while not (repo_root / "src" / "et_downscaling").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))

ee.Initialize(project="ee-change")

print("Repository:", repo_root)
print("Earth Engine ready")

Repository: E:\1SIG22\Varios\github\git_github\26-et-downscaling-fundacion
Earth Engine ready


In [7]:
from et_downscaling.sentinel2 import build_s2_medoid

geometry = ee.Geometry.Point(
    [-74.2, 10.5]
).buffer(500)

image = build_s2_medoid(
    ee.ImageCollection([]),
    geometry,
)

print(
    "Bands:",
    image.bandNames().getInfo(),
)

print(
    "Stats:",
    image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geometry,
        scale=20,
    ).getInfo(),
)

Bands: ['Blue', 'Green', 'Red', 'RedEdge1', 'RedEdge2', 'RedEdge3', 'NIR', 'SWIR1', 'SWIR2']
Stats: {'Blue': None, 'Green': None, 'NIR': None, 'Red': None, 'RedEdge1': None, 'RedEdge2': None, 'RedEdge3': None, 'SWIR1': None, 'SWIR2': None}


In [8]:
from et_downscaling.modis import (
    build_modis_inputs,
)

from et_downscaling.sentinel1 import (
    get_sentinel1_collection,
)

from et_downscaling.sentinel2 import (
    get_sentinel2_collection,
)

from et_downscaling.dataset import (
    build_availability_table,
    get_extraction_observations,
    get_valid_observations,
)


modis_inputs = build_modis_inputs()

station_footprints = (
    modis_inputs["station_footprints"]
)

s2_collection = (
    get_sentinel2_collection(
        station_footprints
    )
)

s1_collection = (
    get_sentinel1_collection(
        station_footprints
    )
)


# Use only one quarter for this diagnostic test.
test_modis_inputs = dict(modis_inputs)

test_modis_inputs["collection"] = (
    modis_inputs["collection"]
    .filterDate(
        "2022-04-01",
        "2022-07-01",
    )
)


availability = (
    build_availability_table(
        test_modis_inputs,
        s2_collection,
        s1_collection,
    )
)

legacy = (
    get_valid_observations(
        availability
    )
)

extraction = (
    get_extraction_observations(
        availability
    )
)


summary = ee.Dictionary(
    {
        "availability_rows":
            availability.size(),

        "legacy_99_9_rows":
            legacy.size(),

        "new_extraction_rows":
            extraction.size(),
    }
).getInfo()

summary

{'availability_rows': 55, 'legacy_99_9_rows': 15, 'new_extraction_rows': 39}

In [9]:
coverage_summary = ee.Dictionary(
    {
        "s2":
            extraction.aggregate_stats(
                "s2_union_coverage_pct"
            ),

        "s1":
            extraction.aggregate_stats(
                "s1_union_coverage_pct"
            ),
    }
).getInfo()

coverage_summary

{'s1': {'max': 100,
  'mean': 71.7948717948718,
  'min': 0,
  'sample_sd': 45.58807524548929,
  'sample_var': 2078.2726045883937,
  'sum': 2800,
  'sum_sq': 280000,
  'total_count': 39,
  'total_sd': 44.99981737124165,
  'total_var': 2024.9835634451017,
  'valid_count': 39,
  'weight_sum': 39,
  'weighted_sum': 2800},
 's2': {'max': 100,
  'mean': 74.65985261612694,
  'min': 0,
  'sample_sd': 39.87638428632919,
  'sample_var': 1590.1260237510019,
  'sum': 2911.734252028951,
  'sum_sq': 277814.43901634816,
  'total_count': 39,
  'total_sd': 39.36182873804951,
  'total_var': 1549.3535616035404,
  'valid_count': 39,
  'weight_sum': 39,
  'weighted_sum': 2911.734252028951}}

In [10]:
print(
    "S1 coverage histogram:"
)

print(
    extraction
    .aggregate_histogram(
        "s1_union_coverage_pct"
    )
    .getInfo()
)


print(
    "\nS2 coverage values:"
)

s2_coverage_values = sorted(
    extraction
    .aggregate_array(
        "s2_union_coverage_pct"
    )
    .getInfo()
)

for value in s2_coverage_values:
    print(
        round(value, 2)
    )
    

S1 coverage histogram:
{'0.0': 11, '100.0': 28}

S2 coverage values:
0
0
0
0
0
3.74
7.66
8.0
19.62
67.7
72.74
77.52
82.84
90.78
94.75
95.14
95.79
97.92
98.5
99.05
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100


In [11]:
coverage_counts = ee.Dictionary(
    {
        "modis_good":
            extraction.size(),

        "s1_ge_99_9":
            extraction.filter(
                ee.Filter.gte(
                    "s1_union_coverage_pct",
                    99.9,
                )
            ).size(),

        "s2_ge_99_9":
            extraction.filter(
                ee.Filter.gte(
                    "s2_union_coverage_pct",
                    99.9,
                )
            ).size(),

        "both_ge_99_9":
            extraction
            .filter(
                ee.Filter.gte(
                    "s1_union_coverage_pct",
                    99.9,
                )
            )
            .filter(
                ee.Filter.gte(
                    "s2_union_coverage_pct",
                    99.9,
                )
            )
            .size(),
    }
).getInfo()

coverage_counts

{'both_ge_99_9': 15, 'modis_good': 39, 's1_ge_99_9': 28, 's2_ge_99_9': 19}

In [12]:
station_counts = {
    "extraction": (
        extraction
        .aggregate_histogram(
            "station"
        )
        .getInfo()
    ),

    "legacy_99_9": (
        legacy
        .aggregate_histogram(
            "station"
        )
        .getInfo()
    ),
}

station_counts


{'extraction': {'Bananera': 7,
  'Bosque seco': 8,
  'Manglar': 7,
  'Palma': 7,
  'Pastos limpios': 10},
 'legacy_99_9': {'Bananera': 1,
  'Bosque seco': 5,
  'Manglar': 3,
  'Palma': 2,
  'Pastos limpios': 4}}

In [13]:
s1_diagnostic = (
    extraction
    .select(
        [
            "station",
            "period_start",
            "s1_products_total",
            "s1_dates_total",
            "s1_union_coverage_pct",
        ]
    )
    .sort(
        "period_start"
    )
)

s1_rows = (
    s1_diagnostic
    .getInfo()["features"]
)

for feature in s1_rows:
    properties = feature["properties"]

    print(
        properties["station"],
        properties["period_start"],
        "products:",
        properties["s1_products_total"],
        "dates:",
        properties["s1_dates_total"],
        "coverage:",
        round(
            properties[
                "s1_union_coverage_pct"
            ],
            2,
        ),
    )

Pastos limpios 2022-04-07 products: 1 dates: 1 coverage: 100
Palma 2022-04-07 products: 1 dates: 1 coverage: 100
Bananera 2022-04-07 products: 1 dates: 1 coverage: 100
Manglar 2022-04-07 products: 1 dates: 1 coverage: 100
Bosque seco 2022-04-07 products: 1 dates: 1 coverage: 100
Pastos limpios 2022-04-15 products: 1 dates: 1 coverage: 100
Bananera 2022-04-15 products: 1 dates: 1 coverage: 100
Bosque seco 2022-04-15 products: 1 dates: 1 coverage: 100
Pastos limpios 2022-04-23 products: 0 dates: 0 coverage: 0
Bosque seco 2022-04-23 products: 0 dates: 0 coverage: 0
Pastos limpios 2022-05-01 products: 1 dates: 1 coverage: 100
Palma 2022-05-01 products: 1 dates: 1 coverage: 100
Manglar 2022-05-01 products: 1 dates: 1 coverage: 100
Bosque seco 2022-05-01 products: 1 dates: 1 coverage: 100
Pastos limpios 2022-05-17 products: 0 dates: 0 coverage: 0
Bananera 2022-05-17 products: 0 dates: 0 coverage: 0
Manglar 2022-05-17 products: 0 dates: 0 coverage: 0
Bosque seco 2022-05-17 products: 0 dates: 